In [ ]:
# Task 1: Data Augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
import matplotlib.pyplot as plt
import numpy as np

# 1. Setup the generator with transformations
datagen = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 2. Load a sample image (update 'sample_food.jpg' with your path)
img = load_img('sample_food.jpg') 
x = img_to_array(img)
x = x.reshape((1,) + x.shape)

# 3. Visualize 5 augmented versions
plt.figure(figsize=(15, 5))
i = 0
for batch in datagen.flow(x, batch_size=1):
    plt.subplot(1, 5, i + 1)
    plt.imshow(img_to_array(batch[0]).astype('uint8'))
    plt.axis('off')
    i += 1
    if i >= 5:
        break
plt.show()

In [ ]:
# Task 2: VGG16 Feature Extraction
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Load VGG16 without top layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Create a generator (just for loading, no augmentation here)
datagen = ImageDataGenerator(rescale=1./255)
generator = datagen.flow_from_directory(
    'sneaker_dataset/', 
    target_size=(224, 224), 
    batch_size=30, 
    class_mode=None, 
    shuffle=False
)

# Extract features
features = base_model.predict(generator)
np.save('sneaker_features.npy', features)
print(f"Features saved with shape: {features.shape}")

In [ ]:
# Task 3: Fine-tuning ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

# Load ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze only the last 20 layers
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Build head
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid') # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# model.fit(...) # Train on your 40-image glasses dataset

In [ ]:
# AI-Generated snippet
from tensorflow.keras.applications import MobileNetV2
model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(160, 160, 3))
x = layers.GlobalAveragePooling2D()(model.output)
outputs = layers.Dense(3, activation='softmax')(x)
model = models.Model(inputs=model.input, outputs=outputs)

In [ ]:
# Modified for Headphones
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Assuming dataset classes are ['over_ear', 'in_ear', 'bone_conduction']
num_classes = 3 

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(160, 160, 3))
base_model.trainable = False # Keep base frozen for feature extraction

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Now call model.fit(train_ds)